In [2]:
import json 
import pandas as pd 
import os 

In [3]:
def generate_benchmark_table(directory_path: str, 
                             dataset_name: str,
                             output_path: str = None):
    data_list = []
    file_paths = [f for f in os.listdir(directory_path) if dataset_name in f and f.endswith('.json')]
    
    for file_path in file_paths:
        model_name = file_path.split('-')[0]
        file_path = os.path.join(directory_path, file_path)
        try:
            with open(file_path, 'r') as f:
                content = json.load(f)
                
            if 'aggregates' in content:
                agg_data = content['aggregates']
                agg_data['model_name'] = model_name
                data_list.append(agg_data)
        except Exception as e:
            print(f"Error reading {file_path}: {e}")

    if not data_list:
        print("No valid JSON files with 'aggregates' found.")
        return

    df = pd.DataFrame(data_list)
    
    df_grouped = df.groupby('model_name').mean().reset_index()
    
    columns_order = [
        'model_name', 
        'mean_volumetric_dice', 
        'mean_surface_dice', 
        'mean_hausdorff95', 
        'mean_masd', 
        'rmse'
    ]
    
    df_final = df_grouped[columns_order].set_index('model_name')
    
    md_table = df_final.to_markdown()
    
    print("### Benchmark Results\n")
    print(md_table)
    if output_path:
        df.to_csv(output_path, index=True)

In [13]:
generate_benchmark_table(directory_path="../benchmarking_results", dataset_name="panther")

### Benchmark Results

| model_name           |   mean_volumetric_dice |   mean_surface_dice |   mean_hausdorff95 |   mean_masd |    rmse |
|:---------------------|-----------------------:|--------------------:|-------------------:|------------:|--------:|
| mr_segmentator       |             0.817645   |           0.918493  |           11.5811  |     1.82541 | 38493.5 |
| mri_segmenter        |             0.796488   |           0.923081  |           12.9455  |     2.0095  | 29448.3 |
| mri_segmenter0       |             0.796154   |           0.92136   |           13.1898  |     2.02767 | 29100.1 |
| mri_segmenter1       |             0.795457   |           0.92206   |           12.933   |     2.04953 | 29716.7 |
| mri_segmenter2       |             0.794414   |           0.920105  |           13.1079  |     2.0212  | 29216.8 |
| mri_segmenter3       |             0.792031   |           0.920704  |           13.1063  |     2.04918 | 29410.5 |
| mri_segmenter4       |             0.79